# 06 — Visualization Functions

**Topic**: All 8 plot functions — one cell each, tweakable parameters, plotly fallback.

**Reference**: NLvib visualization module; MATLAB NLvib plot conventions

**Estimated runtime**: < 10 seconds

## Setup

All plot functions in `nlvib.visualization.plots` share the same design rules:

- Return `matplotlib.figure.Figure` — never call `plt.show()` or mutate global state.
- Accept optional `ax=` to plot into an existing axes.
- Accept optional `backend='matplotlib'|'plotly'` (plotly is an optional dependency).
- **Display**: the figure object itself (last line in a cell) renders inline in Jupyter.

In [ ]:
import sys
sys.path.insert(0, '../src')

import matplotlib
matplotlib.use('Agg')

import numpy as np
import matplotlib.pyplot as plt
from dataclasses import dataclass

from nlvib.visualization.plots import (
    plot_frf, plot_backbone, plot_time_series, plot_phase_portrait,
    plot_floquet, plot_mode_shape, plot_harmonic_content, plot_convergence
)

# Synthetic data for fast execution
@dataclass
class MockResult:
    omega: np.ndarray
    amplitude: np.ndarray
    stability: np.ndarray

print("All plot functions imported.")

### 1. `plot_frf` — Frequency Response Function

In [ ]:
# Synthetic Duffing-like FRF (S-curve shape)
omega_frf = np.linspace(0.6, 1.4, 200)
# Simulate stable/unstable branch
amp_frf = 0.5 / np.sqrt((1 - omega_frf**2)**2 + (0.02*omega_frf)**2)
stability_frf = np.ones(200, dtype=bool)
stability_frf[80:130] = False  # unstable segment between fold points

dof = 0         # ← try changing this
harmonic = 1    # ← try changing this

result_frf = MockResult(omega=omega_frf, amplitude=amp_frf, stability=stability_frf)
fig = plot_frf(result_frf, dof=dof, harmonic=harmonic)
fig.axes[0].set_title(f'FRF — DOF {dof}, harmonic {harmonic}')
fig

### 2. `plot_backbone` — Backbone Curve (NMA)

In [ ]:
# Hardening backbone: frequency increases with amplitude
modal_amp = np.linspace(0.0, 1.0, 100)
omega_bb = 1.0 + 0.3 * modal_amp**2  # hardening
stability_bb = np.ones(100, dtype=bool)

result_bb = MockResult(omega=omega_bb, amplitude=modal_amp, stability=stability_bb)
fig = plot_backbone(result_bb)
fig

### 3. `plot_time_series` — Displacement and Velocity

In [ ]:
omega_ts = 1.02
T_ts = 2*np.pi / omega_ts
t = np.linspace(0, 2*T_ts, 256)
q = 0.3 * np.cos(omega_ts * t) + 0.02 * np.cos(3*omega_ts * t)  # fundamental + 3rd harmonic
dq = -0.3 * omega_ts * np.sin(omega_ts * t) - 0.06 * omega_ts * np.sin(3*omega_ts * t)

fig = plot_time_series(t, q, dq=dq, dof=0)
fig

### 4. `plot_phase_portrait` — Phase Plane

In [ ]:
# Closed orbit in phase plane (periodic motion)
fig = plot_phase_portrait(t, q, dq, dof=0)
fig

### 5. `plot_floquet` — Floquet Multipliers

In [ ]:
# Stable system: all Floquet multipliers inside unit circle
# 2-DOF system has 4 multipliers
decay = 0.92
multipliers = decay * np.array([
    np.exp(1j * 0.8),
    np.exp(-1j * 0.8),
    np.exp(1j * 2.1),
    np.exp(-1j * 2.1),
])

fig = plot_floquet(multipliers)
fig.axes[0].set_title('Floquet multipliers — stable orbit (all inside unit circle)')
fig

### 6. `plot_mode_shape` — Spatial Mode Shape

In [ ]:
# Clamped-free beam: first mode shape (cantilever deflection)
n_nodes = 21
nodes = np.linspace(0, 1, n_nodes)
# Approximate first mode shape: psi(x) = 1 - cos(pi*x/2)
displacement = 1 - np.cos(np.pi * nodes / 2)

fig = plot_mode_shape(nodes, displacement, title='Mode 1 — clamped-free beam')
fig

### 7. `plot_harmonic_content` — Fourier Amplitude Spectrum

In [ ]:
harmonic_num = 3   # ← try changing this (which harmonic to highlight)

# Harmonic amplitudes for Duffing at resonance: decaying spectrum
Q_harmonics = np.array([0.312, 0.0, 0.008, 0.0, 0.0003])

fig = plot_harmonic_content(Q_harmonics, omega=1.02)
fig

### 8. `plot_convergence` — Residual Convergence

In [ ]:
# Simulate Newton convergence: quadratic decay
residuals = [1.0]
for _ in range(8):
    residuals.append(residuals[-1]**2 * 0.5)
residuals = np.array(residuals)

fig = plot_convergence(residuals)
fig

In [ ]:
# Plotly backend — graceful fallback if not installed
try:
    fig_plotly = plot_frf(result_frf, backend='plotly')
    # Access the plotly figure via the hidden attribute
    if hasattr(fig_plotly, '_nlvib_plotly_fig'):
        print("Plotly figure created successfully.")
        print("To display: fig_plotly._nlvib_plotly_fig.show()")
    fig_plotly
except ImportError as e:
    print(f"Plotly not installed: {e}")
    print("Install with: pip install plotly")
    print("Falling back to matplotlib backend.")
    fig_plotly = plot_frf(result_frf, backend='matplotlib')
    fig_plotly

## Key Takeaways

- All 8 plot functions return a `Figure` — display by putting the figure as the last expression in a cell.
- Never call `plt.show()` — it blocks the kernel. The figure object renders automatically in Jupyter.
- The `stability` array controls stable (solid) vs. unstable (dashed) branch rendering.
- `backend='plotly'` gives interactive zoom/pan — requires `pip install plotly`; falls back gracefully.